# RA-NAS — Kaggle Run Notebook
**Reasoning-Agent Neural Architecture Search on CIFAR-10**

This notebook clones the RA-NAS repository, installs dependencies, configures the Groq API key via Kaggle Secrets, and runs the full iterative NAS experiment on a Kaggle GPU.

> **Before running:** Add your Groq API key as a Kaggle Secret named `GROQ_API_KEY`  
> Notebook → Add-ons → Secrets → New Secret → Name: `GROQ_API_KEY`

**Recommended runtime:** GPU T4 x1 (free tier) — uses ~2–4 GB VRAM, 20–40 min for 10 iterations.

## Step 1 — Verify GPU

In [ ]:
import torch

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM            : {vram_gb:.1f} GB')
else:
    print('WARNING: No GPU detected. Enable GPU under Notebook settings → Accelerator.')


## Step 2 — Clone Repository & Install Dependencies

In [ ]:
import os

REPO_URL = 'https://github.com/GyanendraChaubey/RA-NAS-Code.git'  # <-- replace with your repo URL
REPO_DIR = '/kaggle/working/RA-NAS'

if not os.path.exists(REPO_DIR):
    os.system(f'git clone {REPO_URL} {REPO_DIR}')
    print('Repository cloned.')
else:
    os.system(f'git -C {REPO_DIR} pull')
    print('Repository updated.')


In [ ]:
# Install project dependencies (torch/torchvision already in Kaggle base image)
os.system('pip install -q openai pyyaml tqdm python-dotenv')
print('Dependencies installed.')


In [ ]:
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')


## Step 3 — Load Groq API Key from Kaggle Secrets

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['GROQ_API_KEY'] = UserSecretsClient().get_secret('GROQ_API_KEY')
    print('API key loaded from Kaggle Secrets.')
except Exception as e:
    print(f'Could not load from Kaggle Secrets: {e}')
    print('Falling back to mock_mode=True (no real LLM calls).')


## Step 4 — Configure the Experiment

Adjust the settings below to control the NAS run.

| Parameter | Default | Notes |
|-----------|---------|-------|
| `NUM_ITERATIONS` | 10 | NAS iterations (propose → train → evaluate cycles) |
| `EPOCHS_PER_ARCH` | 10 | Training epochs per candidate architecture |
| `BATCH_SIZE` | 256 | Larger is faster on T4 GPU |
| `MOCK_MODE` | `False` | Set `True` to skip real LLM calls (for testing) |
| `EXPLORE_EVERY` | 3 | Force fresh random proposal every N iterations |

In [ ]:
# ── Experiment hyper-parameters ──────────────────────────────────────────────
NUM_ITERATIONS   = 20     # number of NAS iterations
EPOCHS_PER_ARCH  = 50     # training epochs per candidate
BATCH_SIZE       = 256    # batch size (T4 handles 256 comfortably)
MOCK_MODE        = False  # True = no LLM API calls, uses random proposals
EXPLORE_EVERY    = 2      # force fresh proposal every N iterations (0 = disable)
SEED             = 42
# ─────────────────────────────────────────────────────────────────────────────


## Step 5 — Build Config Dictionaries

In [ ]:
import random
import numpy as np

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

train_config = {
    'dataset': {
        'name': 'cifar10',
        'data_dir': '/kaggle/working/data',
        'num_classes': 10,
        'val_split': 0.1,
    },
    'training': {
        'epochs': EPOCHS_PER_ARCH,
        'batch_size': BATCH_SIZE,
        'learning_rate': 0.1,
        'momentum': 0.9,
        'weight_decay': 5e-4,
        'optimizer': 'sgd',
        'scheduler': 'cosine_warmup',
        'warmup_epochs': 5,
        'screening_epochs': 5,
        'num_workers': 2,
        'seed': SEED,
        'augmentation': {
            'cutout': True,
            'cutout_length': 16,
            'mixup': True,
            'mixup_alpha': 0.2,
        },
    },
    'early_stopping': {
        'enabled': True,
        'patience': 5,
        'monitor': 'val_accuracy',
        'mode': 'max',
    },
    'experiment': {
        'name': 'kaggle_run',
        'output_dir': '/kaggle/working/experiments',
        'save_best_only': True,
    },
    'architecture_constraints': {
        'min_layers': 2,
        'max_layers': 8,
        'min_filters': 16,
        'max_filters': 512,
        'allowed_activations': ['relu', 'gelu', 'silu'],
        'allowed_kernels': [3, 5, 7],
    },
}

agent_config = {
    'llm': {
        'provider': 'groq',
        'model': 'llama-3.3-70b-versatile',
        'base_url': 'https://api.groq.com/openai/v1',
        'temperature': 1.2,
        'temperature_min': 0.7,
        'temperature_decay': 0.05,
        'max_tokens': 1536,
        'api_key_env': 'GROQ_API_KEY',
    },
    'agent': {
        'max_iterations': NUM_ITERATIONS,
        'top_k_memory': 5,
        'retry_on_invalid': 3,
        'feedback_strategy': 'top_k',
        'mock_mode': MOCK_MODE,
        'diversity_penalty': True,
    },
}

from src.utils.config_loader import merge_configs
merged_config = merge_configs(train_config, agent_config)
print('Config ready.')


## Step 6 — Download CIFAR-10 Dataset

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

DATA_DIR = '/kaggle/working/data'
os.makedirs(DATA_DIR, exist_ok=True)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])
eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

full_train_aug  = datasets.CIFAR10(DATA_DIR, train=True,  download=True,  transform=train_transform)
full_train_eval = datasets.CIFAR10(DATA_DIR, train=True,  download=False, transform=eval_transform)

total_size = len(full_train_aug)       # 50 000
val_size   = int(total_size * 0.1)    # 5 000
train_size = total_size - val_size    # 45 000

gen     = torch.Generator().manual_seed(SEED)
indices = torch.randperm(total_size, generator=gen).tolist()

train_loader = DataLoader(
    Subset(full_train_aug,  indices[:train_size]),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True,
)
val_loader = DataLoader(
    Subset(full_train_eval, indices[train_size:]),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True,
)
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')


## Step 7 — Initialise Agent, Generator & Controller

In [ ]:
import logging
from pathlib import Path
from datetime import datetime

from src.agents.llm_agent import LLMAgent
from src.agents.memory import ExperimentMemory
from src.evaluation.evaluator import Evaluator
from src.nas.architecture_generator import ArchitectureGenerator
from src.nas.controller import NASController
from src.nas.search_space import SEARCH_SPACE
from src.training.trainer import Trainer
from src.utils.logger import get_logger

# ── Ensure all sub-module INFO logs (LLM reasoning, prediction errors, etc.)
#    appear in notebook cell output. BasicConfig configures the root logger so
#    loggers like src.agents.llm_agent propagate their messages here.
logging.basicConfig(
    level=logging.INFO,
    format='%(levelname)s | %(message)s',
    force=True,   # override any pre-existing root-logger config from imports
)

timestamp      = datetime.now().strftime('%Y%m%d_%H%M%S')
experiment_dir = Path(f'/kaggle/working/experiments/kaggle_run_{timestamp}')
experiment_dir.mkdir(parents=True, exist_ok=True)

logger = get_logger('ra_nas', str(experiment_dir))

constraints = merged_config['architecture_constraints']
generator   = ArchitectureGenerator(SEARCH_SPACE, constraints, seed=SEED)
memory      = ExperimentMemory()
agent       = LLMAgent(agent_config=merged_config, search_space=generator, memory=memory)

def trainer_factory(model, iteration_dir, override_config=None):
    cfg = override_config if override_config is not None else merged_config
    return Trainer(model=model, config=cfg, device=DEVICE,
                   experiment_dir=iteration_dir, logger=logger)

def evaluator_factory(model):
    return Evaluator(model=model, device=DEVICE)

controller = NASController(
    agent=agent, generator=generator,
    trainer=trainer_factory, evaluator=evaluator_factory,
    memory=memory, config=merged_config,
    train_loader=train_loader, val_loader=val_loader,
    device=DEVICE, logger=logger,
    experiment_dir=experiment_dir,
    num_classes=10,
    explore_every=EXPLORE_EVERY,
)
print('Controller ready.')


## Step 8 — Run the NAS Loop
Each iteration: **propose** architecture → **train** → **evaluate** → store in **memory** → **refine**.

In [ ]:
import json

results = controller.run(num_iterations=NUM_ITERATIONS)

memory.save(str(experiment_dir / 'memory.json'))
with (experiment_dir / 'metrics.json').open('w') as f:
    json.dump(results, f, indent=2)

print(f'Done. Results saved to {experiment_dir}')


## Step 9 — Results Summary Table

In [ ]:
header = f"{'Iter':>4} | {'Layers':>6} | {'Filters':<30} | {'Act':<5} | {'Pool':<4} | {'Val Acc':>8} | {'Params':>10} | {'FLOPs':>12}"
print(header)
print('-' * len(header))
for r in results:
    a, m = r['arch'], r['metrics']
    print(
        f"{r['iteration']:>4} | {a['num_layers']:>6} | {str(a['filters']):<30} | "
        f"{a['activation']:<5} | {a['pooling']:<4} | "
        f"{m['val_accuracy']:>7.2f}% | {int(m['num_params']):>10,} | {int(m['flops']):>12,}"
    )


## Step 10 — Plot Val Accuracy Across Iterations

In [ ]:
import matplotlib.pyplot as plt

iterations = [r['iteration']                for r in results]
val_accs   = [r['metrics']['val_accuracy']   for r in results]
train_accs = [r['metrics']['train_accuracy']  for r in results]
params     = [r['metrics']['num_params']      for r in results]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(iterations, val_accs,   'o-',  label='Val Accuracy',   color='royalblue')
axes[0].plot(iterations, train_accs, 's--', label='Train Accuracy', color='coral', alpha=0.7)
axes[0].set_xlabel('NAS Iteration')
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_title('Accuracy per NAS Iteration')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

sc = axes[1].scatter(params, val_accs, c=iterations, cmap='viridis', s=80, zorder=3)
for r in results:
    axes[1].annotate(
        str(r['iteration']),
        (r['metrics']['num_params'], r['metrics']['val_accuracy']),
        textcoords='offset points', xytext=(4, 4), fontsize=8,
    )
plt.colorbar(sc, ax=axes[1], label='Iteration')
axes[1].set_xlabel('Parameter Count')
axes[1].set_ylabel('Val Accuracy (%)')
axes[1].set_title('Accuracy vs Model Size')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(experiment_dir / 'nas_results.png'), dpi=150, bbox_inches='tight')
plt.show()


## Step 11 — Best Architecture Found

In [ ]:
best = max(results, key=lambda r: r['metrics']['val_accuracy'])

print('=' * 55)
print(f"  Best NAS iteration : {best['iteration']}")
print('=' * 55)
print(f"  Val accuracy       : {best['metrics']['val_accuracy']:.4f}%")
print(f"  Train accuracy     : {best['metrics']['train_accuracy']:.4f}%")
print(f"  Val loss           : {best['metrics']['val_loss']:.4f}")
print(f"  Num parameters     : {int(best['metrics']['num_params']):,}")
print(f"  FLOPs (est.)       : {int(best['metrics']['flops']):,}")
print(f"  Inference latency  : {best['metrics']['inference_time_ms']:.2f} ms")
print()
print('  Architecture:')
for k, v in best['arch'].items():
    print(f'    {k:<22}: {v}')
print('=' * 55)


## Step 12 — Package Outputs for Download
Zips the entire experiment directory (checkpoints, memory, metrics, plot) into `/kaggle/working/` for easy download via the Output panel.

In [ ]:
import shutil

archive_path = shutil.make_archive(
    base_name=str(experiment_dir),
    format='zip',
    root_dir=experiment_dir.parent,
    base_dir=experiment_dir.name,
)
size_mb = os.path.getsize(archive_path) / 1e6
print(f'Archive: {archive_path}  ({size_mb:.1f} MB)')
print('Download it from the Kaggle Output panel on the right.')
